# 04 Win Rate — Logistic Regression

## Goal

For every `(character, card)` pair in `gold_card_choice_events`, restrict to occasions where the card was offered and fit

```
victory ~ was_picked + hp_ratio + floor + relic_count + ascension_level
```

`was_picked`'s coefficient is the card's effect on win probability *holding run state at the time of the offer constant* — a cleaner signal than the raw pick/no-pick lift computed in 03, which doesn't control for anything.

This fits every card directly rather than pre-selecting a curated "cards of interest" list by raw lift and then re-estimating on the same data. That selection step is a winner's-curse setup: any card's raw lift is true effect plus sampling noise, and sorting by it preferentially keeps cards that got a lucky noise draw — that noise doesn't average out on a second look at the same rows, so the regression would end up biased toward whatever the screen happened to reward. Fitting every card sidesteps the problem entirely instead of working around it with a data split. `03`'s raw-lift screen is still worth checking against the results here as a sanity check (a card with strong raw lift but a controlled odds ratio near 1 suggests the raw lift was confounded), just not as a required input.

**Deliberately excluded:** `floor_reached` and `floors_gained` are not covariates here, even though they're in the table. Both are facts about how the run *ended*, not facts known at the time of the pick — `floor_reached` is close to deterministic of `victory` (a run that reaches floor 57 essentially won), so including it would leak the outcome into the predictors rather than control for a legitimate confounder. `floor` (the pick's own floor, i.e. `choice_floor`) is fine to include — that's "how far into the run this decision happened," known at decision time.

In [ ]:
import sys
from pathlib import Path

# Jupyter's cwd is the notebook's directory; project root is one level up. analysis/ isn't
# pip-installed (no editable install in this venv), so it has to be added to sys.path
# explicitly for `import analysis` to work from a notebook - Dagster gets this for free by
# always running from the project root.
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

GOLD_CARD_CHOICE_EVENTS_PATH = str(PROJECT_ROOT / "raw_data" / "gold" / "card_choice_events")
REGRESSION_INPUT_PARQUET = str(PROJECT_ROOT / "raw_data" / "win_rate_regression_input.parquet")

In [ ]:
from analysis.win_rate_modeling import collect_win_rate_regression_input

regression_pd = collect_win_rate_regression_input(
    PROJECT_ROOT, GOLD_CARD_CHOICE_EVENTS_PATH, REGRESSION_INPUT_PARQUET
)

In [ ]:
from analysis.win_rate_modeling import fit_all_card_logits, load_win_rate_regression_input

regression_pd = load_win_rate_regression_input(REGRESSION_INPUT_PARQUET)

FORMULA = "victory ~ was_picked + hp_ratio + floor + relic_count + ascension_level"
MIN_REGRESSION_ROWS = 200

results_pd = fit_all_card_logits(regression_pd, FORMULA, MIN_REGRESSION_ROWS)
n_fitted = results_pd["error"].isna().sum()
n_converged = (results_pd["error"].isna() & results_pd["converged"]).sum()
print("Fitted:", n_fitted, "of", len(results_pd))
print("Converged:", n_converged, "of", n_fitted, "fitted")

In [2]:
print("Error breakdown (top 20):")
print(results_pd["error"].value_counts(dropna=False).head(20))
print()
print("Convergence among fitted models:")
print(results_pd.loc[results_pd["error"].isna(), "converged"].value_counts(dropna=False))

Error breakdown (top 20):
error
NaN                2231
too few rows        896
Singular matrix      12
Name: count, dtype: int64

Convergence among fitted models:
converged
True     2221
False      10
Name: count, dtype: Int64


In [ ]:
# Prismatic Shard (a relic) makes every reward screen show one card from each class instead of
# three from your own - so an IRONCLAD run can genuinely be offered and pick a card whose home
# class is THE_SILENT. Those off-class picks are real rows, not a data bug, but they're rare
# enough (and confounded with "this run had Prismatic Shard", which nothing here controls for)
# that they don't belong in the same table as a card's native-class effect.
#
# The split is clean in the data itself: for a base card (upgraded and base versions summed
# together), the character who "owns" it holds the overwhelming majority of its total offers;
# every other character sits at a sliver. Checked empirically across the full results - character
# shares cluster at <0.5% (off-class/Prismatic Shard, or one-off modded-content cards) or >15%
# (native class, or a colorless card split roughly by how often each character gets played) with
# nothing in between, so any threshold in that gap works; 5% is just a round number inside it.
results_pd["base_card"] = results_pd["card_name"].str.replace(r"\+1$", "", regex=True)
character_n = results_pd.groupby(["base_card", "character_chosen"])["n"].transform("sum")
base_card_total_n = results_pd.groupby("base_card")["n"].transform("sum")
results_pd["character_share"] = character_n / base_card_total_n

NATIVE_SHARE_THRESHOLD = 0.05
results_pd["native_pool"] = results_pd["character_share"] >= NATIVE_SHARE_THRESHOLD
print("Native-pool rows:", results_pd["native_pool"].sum(), "of", len(results_pd))

### Results

`odds_ratio` > 1 means picking the card is associated with higher win odds after controlling for HP ratio, floor, relic count, and ascension at the time it was offered; < 1 means lower.

The `significant` table below is restricted four ways, each guarding against a different way "significant" can be misleading at this scale:

- **`converged`** — drops models where the MLE didn't converge. Small, lopsided card groups (a card picked in nearly every winning run and almost never in a losing one) can hit quasi/complete separation, where the optimizer keeps pushing `was_picked`'s coefficient toward infinity without the likelihood actually converging. statsmodels doesn't raise for this — it returns whatever the optimizer had on its last iteration, with a wide, unstable confidence interval (an odds ratio of 22 with a 95% CI of roughly [1, 400] is the signature of this, not a real effect). `model.mle_retvals["converged"]` is the flag that catches it.
- **`native_pool`** — drops off-class cards. Prismatic Shard (a relic) makes every reward screen show one card from each class instead of three from your own, so e.g. an IRONCLAD run can pick a card whose home class is THE_SILENT. Those picks are real, but rare and confounded with "this run had Prismatic Shard" (uncontrolled for here), so they're excluded rather than reported as if they were a native-class effect. The threshold is empirical, not assumed: aggregating each base card's offers by character, shares cluster at <0.5% (off-class, or one-off modded-content cards that leaked into the raw data) or >15% (native class, or a colorless card split roughly by how often each character gets played), with nothing in between — any cutoff in that gap works.
- **`was_picked_qvalue < 0.05`** (Benjamini-Hochberg FDR-adjusted p-value) instead of the raw `was_picked_pvalue` — with 2,232 independent regressions, filtering on the raw p-value at 0.05 would let roughly 5% of cards with no true effect (100+ cards) cross significance by chance alone. BH correction controls the expected false-discovery proportion among the reported results instead.
- **effect size outside `[0.9, 1.1]`** — clearing the q-value bar alone doesn't rule out a statistically real but practically trivial effect (an odds ratio of 1.02 estimated from hundreds of thousands of rows is technically significant and not worth acting on). Requiring the odds ratio to actually move the needle keeps the table focused on effects worth caring about, not just ones large samples can detect.

Cards that don't clear all four bars are dropped from this view but still available in `results_pd`, along with `pseudo_r2` and `auc` for judging each surviving model's overall fit (`auc` is computed in-sample, so it reads as fit quality, not held-out predictive performance).

In [ ]:
significant = results_pd[
    results_pd["error"].isna()
    & results_pd["converged"].fillna(False)
    & results_pd["native_pool"]
    & (results_pd["was_picked_qvalue"] < 0.05)
    & ((results_pd["odds_ratio"] <= 0.9) | (results_pd["odds_ratio"] >= 1.1))
].sort_values(["character_chosen", "odds_ratio"], ascending=[True, False])

cols = ["character_chosen", "card_name", "n", "odds_ratio", "odds_ratio_ci_low", "odds_ratio_ci_high", "was_picked_pvalue", "was_picked_qvalue", "pseudo_r2", "auc"]
significant[cols]

In [ ]:
summary = (
    significant.groupby("character_chosen")
    .agg(n_significant=("card_name", "count"),
         n_positive=("odds_ratio", lambda s: (s > 1).sum()),
         n_negative=("odds_ratio", lambda s: (s < 1).sum()))
)
summary.to_string()

In [ ]:
top = (
    significant.groupby("character_chosen")
    .apply(lambda g: g.nlargest(5, "odds_ratio")[["card_name","n","odds_ratio","was_picked_pvalue"]], include_groups=False)
)
bottom = (
    significant.groupby("character_chosen")
    .apply(lambda g: g.nsmallest(5, "odds_ratio")[["card_name","n","odds_ratio","was_picked_pvalue"]], include_groups=False)
)
print("TOP 5 per character (best odds_ratio):")
print(top.to_string())
print()
print("BOTTOM 5 per character (worst odds_ratio):")
print(bottom.to_string())

In [7]:
print(regression_pd.memory_usage(deep=True).sum() / 1e9, "GB in memory")
print(len(regression_pd), "rows")

2.996086615 GB in memory
230466285 rows


In [ ]:
results_pd.to_parquet("../raw_data/card_win_rate_regression_results.parquet", index=False)

In [9]:
significant[cols].to_csv("../raw_data/card_win_rate_significant.csv", index=False)
summary.to_csv("../raw_data/card_win_rate_summary.csv")